# Guide 4 — The semantic layer

This guide examines the JSON-LD that BattINFO produces, explains how it connects to EMMO, and shows how to work with it using RDF tools.

**Estimated time:** 20 minutes
**Prerequisites:** [Guide 2](02-first-cell-type.ipynb) or [Guide 3](03-linked-records.ipynb)

In [1]:
import json
import warnings
from pathlib import Path

SCRATCH = Path("_scratch/guide-04")
SCRATCH.mkdir(parents=True, exist_ok=True)

from battinfo import CellSpec, publish
from battinfo.api import publish_record

cell_spec = CellSpec(
    manufacturer="A123 Systems",
    model="ANR26650M1-B",
    format="cylindrical",
    chemistry="Li-ion",
    positive_electrode_basis="LFP",
    negative_electrode_basis="graphite",
    size_code="R26650",
    properties={
        "nominal_capacity":   {"value": 2.5,  "unit": "Ah"},
        "nominal_voltage":    {"value": 3.3,  "unit": "V"},
        "rated_energy":       {"value": 8.25, "unit": "Wh"},
        "mass":               {"value": 76.0, "unit": "g"},
        "diameter":           {"value": 26.0, "unit": "mm"},
        "height":             {"value": 65.0, "unit": "mm"},
        "internal_resistance":{"value": 6.0,  "unit": "mΩ"},
        "cycle_life":         {"value_text": ">1000", "unit": "count"},
    },
)

result = publish(cell_spec, destination="local", root=SCRATCH)
output = publish_record(
    result.debug_paths["canonical_record_path"],
    target_root=SCRATCH / "resolver",
)

jsonld_path = Path(output["output_dir"], "index.jsonld")
jsonld = json.loads(jsonld_path.read_text(encoding="utf-8"))
print("Ready. IRI:", result.canonical_iri)

Ready. IRI: https://w3id.org/battinfo/spec/wckm-y5a4-he0k-2scd


## The full resolver JSON-LD

In [2]:
print(json.dumps(jsonld, indent=2))

{
  "@context": [
    "https://w3id.org/emmo/domain/battery/context",
    {
      "schema": "https://schema.org/",
      "csvw": "http://www.w3.org/ns/csvw#"
    }
  ],
  "@id": "https://w3id.org/battinfo/spec/wckm-y5a4-he0k-2scd",
  "@type": [
    "BatteryCellSpecification",
    "schema:CreativeWork"
  ],
  "schema:identifier": "wckm-y5a4-he0k-2scd",
  "schema:name": "A123 Systems ANR26650M1-B",
  "schema:manufacturer": {
    "@type": "schema:Organization",
    "schema:name": "A123 Systems"
  },
  "schema:size": "R26650",
  "hasProperty": [
    {
      "@type": [
        "NominalCapacity",
        "ConventionalProperty"
      ],
      "hasNumericalPart": {
        "@type": "RealData",
        "hasNumberValue": 2.5
      },
      "hasMeasurementUnit": "https://w3id.org/emmo#AmpereHour"
    },
    {
      "@type": [
        "NominalVoltage",
        "ConventionalProperty"
      ],
      "hasNumericalPart": {
        "@type": "RealData",
        "hasNumberValue": 3.3
      },
      "hasM

### `@context`

Maps short names to full IRIs. Every term — `BatteryCell`, `hasNumericalPart`, `NominalCapacity` — resolves to a UUID-based IRI in the EMMO domain-battery ontology. BattINFO bundles a local copy so processing works offline.

### `@id`

The permanent IRI for this cell spec, resolvable at `https://w3id.org/battinfo/spec/{uid}`.

### `@type` stacking

In [3]:
print("@type classes:")
for t in jsonld["@type"]:
    print(f"  {t}")

@type classes:
  BatteryCellSpecification
  schema:CreativeWork


The specification node (`BatteryCellSpecification`) carries two types:

| Vocabulary | Type | Rationale |
|---|---|---|
| EMMO | `BatteryCellSpecification` | Information entity describing a class of cells |
| schema.org | `schema:CreativeWork` | The specification document as a creative artifact |

The **physical EMMO classes** derived from cell-spec fields are placed on an `isDescriptionFor` anonymous node:

| Source field | Physical EMMO class (on `isDescriptionFor`) |
|---|---|
| `format: "cylindrical"` | `CylindricalBattery` |
| `chemistry: "Li-ion"` | `LithiumIonBattery` |
| `positive_electrode_basis: "LFP"` | `LithiumIonIronPhosphateBattery` |
| `negative_electrode_basis: "graphite"` | `LithiumIonGraphiteBattery` |
| *(always)* | `BatteryCell` |

Placing physical types on `isDescriptionFor` is semantically correct: the specification is an information entity, not a physical cell. The old pattern (putting `BatteryCell` directly on the specification `@type`) implied the specification IS a physical cell, which is wrong.

No manual annotation — derived entirely from plain-text field values.

### `hasProperty` — the canonical EMMO quantity pattern

### `isDescriptionFor` — where physical EMMO types live

The resolver JSON-LD above is a **lightweight discovery document** — it omits the physical type stacking to keep file size small. The full descriptor JSON-LD (produced by `to_jsonld`) includes an `isDescriptionFor` anonymous node that carries the physical EMMO classes:



This placement is semantically correct: the specification is an *information entity*, not a physical cell. Putting `BatteryCell` on the specification `@type` would assert the specification IS a physical cell — which is wrong.

### `hasDescription` — the reverse link from cell instance to specification

Cell instances (physical objects with serial numbers) carry `hasDescription` pointing to the specification IRI, plus `schema:isVariantOf` for schema.org alignment:



The directionality reflects how data is created: the cell spec is registered first and given a stable IRI; individual cells with serial numbers are created later and reference it. Both `hasDescription` (EMMO) and `schema:isVariantOf` (schema.org) express the same relationship in their respective vocabularies.


In [4]:
# Show isDescriptionFor from the full descriptor pipeline
from battinfo.transform.json_to_jsonld import _descriptor_specification_to_jsonld

spec = {
    "format": "cylindrical",
    "chemistry": "Li-ion",
    "positive_electrode_basis": "LFP",
    "negative_electrode_basis": "graphite",
}
node = _descriptor_specification_to_jsonld(spec)

print("Specification @type:", node["@type"])
print("isDescriptionFor @type:", node["isDescriptionFor"]["@type"])


Specification @type: ['BatteryCellSpecification', 'schema:CreativeWork']
isDescriptionFor @type: ['BatteryCell', 'CylindricalBattery', 'LithiumIonBattery', 'LithiumIonIronPhosphateBattery', 'LithiumIonGraphiteBattery']


In [5]:
print("Quantitative properties:")
for prop in jsonld.get("hasProperty", []):
    types = prop["@type"]
    class_name = types[0] if isinstance(types, list) else types
    value = prop.get("hasNumericalPart", {}).get("hasNumberValue")
    unit  = prop.get("hasMeasurementUnit", "").split("#")[-1]
    print(f"  {class_name}: {value}  ({unit})")

Quantitative properties:
  NominalCapacity: 2.5  (AmpereHour)
  NominalVoltage: 3.3  (Volt)
  NominalEnergy: 8.25  (WattHour)
  Mass: 76.0  (Gram)
  Diameter: 26.0  (MilliMetre)
  Height: 65.0  (MilliMetre)
  InternalResistance: 6.0  (MilliOhm)
  CycleLife: None  (EMMO_5ebd5e01_0ed3_49a2_a30d_cd05cbe72978)


The pattern is:

```
hasProperty → NominalCapacity, ConventionalProperty
                ├── hasNumericalPart → RealData → hasNumericalValue: 2.5
                └── hasMeasurementUnit: <EMMO AmpereHour IRI>
```

This is the **canonical EMMO quantity pattern** — identical across all EMMO domain modules.

## Validating JSON-LD

In [6]:
from battinfo.validate.jsonld import validate_jsonld_report

report = validate_jsonld_report(jsonld)
print("ok:", report.ok)
for issue in report.issues:
    print(f"  [{issue.severity}] {issue.code}: {issue.message}")

ok: True


| Check | What fails |
|---|---|
| `@type` term check | Bare `@type` value not in EMMO context |
| RDF parse | JSON-LD rdflib cannot parse into a valid RDF dataset |
| URDNA2015 | JSON-LD that cannot be canonically normalised |

## Loading into an RDF graph

In [7]:
from rdflib import Graph

g = Graph()
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=DeprecationWarning, module="rdflib")
    g.parse(str(jsonld_path), format="json-ld")

print(f"Triple count: {len(g)}")

Triple count: 62


In [8]:
# All triples in the graph
for s, p, o in sorted(g):
    s_l = str(s).split("/")[-1].split("#")[-1]
    p_l = str(p).split("/")[-1].split("#")[-1]
    o_l = str(o).split("/")[-1].split("#")[-1]
    print(f"  {s_l}  →  {p_l}  →  {o_l}")

  N07cdc0617df84fbe9e02d1cff8d9e701  →  type  →  EMMO_d8aa8e1f_b650_416d_88a0_5118de945456
  N07cdc0617df84fbe9e02d1cff8d9e701  →  type  →  electrochemistry_8abde9d0_84f6_4b4f_a87e_86028a397100
  N07cdc0617df84fbe9e02d1cff8d9e701  →  EMMO_8ef3cd6d_ae58_4a8d_9fc0_ad8f49015cd0  →  N305e542159ae4789966521c41269ee2f
  N07cdc0617df84fbe9e02d1cff8d9e701  →  EMMO_bed1d005_b04e_4a90_94cf_02bc678a8569  →  AmpereHour
  N28e0a161800143c981375372093718b6  →  type  →  EMMO_c1c8ac3c_8a1c_4777_8e0b_14c1f9f9b0c6
  N28e0a161800143c981375372093718b6  →  type  →  EMMO_d8aa8e1f_b650_416d_88a0_5118de945456
  N28e0a161800143c981375372093718b6  →  EMMO_8ef3cd6d_ae58_4a8d_9fc0_ad8f49015cd0  →  N3a8c1a6039db45fda464b3115dba3f00
  N28e0a161800143c981375372093718b6  →  EMMO_bed1d005_b04e_4a90_94cf_02bc678a8569  →  MilliMetre
  N305e542159ae4789966521c41269ee2f  →  type  →  EMMO_18d180e4_5e3e_42f7_820c_e08951223486
  N305e542159ae4789966521c41269ee2f  →  EMMO_faf79f53_749d_40b2_807c_d34244c192f4  →  2.5
  N33b5d5

## Combining multiple records in one graph

In [9]:
import battinfo

ws = battinfo.workspace(root=SCRATCH / "multi")
multi_spec = battinfo.CellSpec(
    manufacturer="A123 Systems", model="ANR26650M1-B",
    format="cylindrical", chemistry="Li-ion",
    positive_electrode_basis="LFP", negative_electrode_basis="graphite",
    properties={"nominal_capacity": {"value": 2.5, "unit": "Ah"}},
)
ws.add("cell", spec=multi_spec, serial_numbers=["a123-demo-multi-001"])
ws.save()

# Publish each saved record as a resolver artifact
from battinfo.api import publish_record as _publish_one

multi_records = SCRATCH / "multi" / ".battinfo" / "records" / "examples"
multi_resolver = SCRATCH / "multi-resolver"
for record_file in sorted(multi_records.rglob("*.json")):
    _publish_one(record_file, target_root=multi_resolver)

jsonld_files = list(multi_resolver.rglob("index.jsonld"))
g2 = Graph()
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", category=DeprecationWarning, module="rdflib")
    for jf in jsonld_files:
        g2.parse(str(jf), format="json-ld")

print(f"Combined graph: {len(g2)} triples from {len(jsonld_files)} files")

  note: using legacy records/examples/ layout (pre-0.8 workspace)
  cell: a123-demo-multi-001  (IRI auto-assigned)
Saved 2 record(s) under <repo>\docs\guides\_scratch\guide-04\multi\.battinfo\records\examples:
  cell spec   A123 Systems ANR26650M1-B            [updated]  .battinfo\records\examples\cell-spec\cell-spec-e4t5-w9re-mpq6-8g7e.json
  cell        a123-demo-multi-001                  [updated]  .battinfo\records\examples\cell-instance\cell-aaqe-rvbd-c0b6-p1k3.json

  Next: ws.list(verbose=True) to inspect, or ws.publish() to publish.


Combined graph: 20 triples from 2 files


## Mapping tables

BattINFO's semantic output is driven by three curated files:

| File | Role |
|---|---|
| `assets/mappings/domain-battery/property_map.curated.json` | JSON key → EMMO class IRI |
| `assets/mappings/domain-battery/unit_map.curated.json` | Unit string → EMMO/QUDT IRI |
| `assets/mappings/domain-battery/entity_type_map.json` | format/chemistry/electrode → @type array |

In [10]:
prop_map = json.loads(
    # Repo files sit two levels up from this notebook.
    Path("../../assets/mappings/domain-battery/property_map.curated.json").read_text(encoding="utf-8")
)
print(f"Property map version: {prop_map['version']}")
print(f"Curated entries: {len(prop_map['mappings'])}")
print()
for entry in prop_map["mappings"][:8]:
    print(f"  {entry['key']:42s} → {entry['class_pref_label']}")
print("  ...")

Property map version: 0.3.0
Curated entries: 66

  ac_internal_resistance                     → ACInternalResistance
  available_volume                           → Volume
  calendar_life                              → CalendarLife
  cap_assembly_weight                        → Mass
  capacity_fade                              → CapacityFade
  certified_usable_energy                    → NominalEnergy
  charging_cutoff_voltage                    → UpperVoltageLimit
  charging_temperature_max                   → MaximumChargingTemperature
  ...


## Next

- **[Guide 5 — Descriptors](05-descriptors.ipynb)**: author research-grade cell descriptors and see richer electrode-level JSON-LD